# RAG 第 19 课：Citations 与 Grounding（可验证的答案）

到目前为止，LLM 输出的答案都是一段文字。
用户并不知道哪一句话来自哪个文档，出错了也没法追溯。

真实产品里你会反复被问到同一个问题：

**"这个结论是哪来的？"**

这节课我们给答案加上**Citations**（引用）：

1. LLM 输出**结构化 JSON**：每一句话都标注它来自哪个 doc_id
2. 我们再用一个**grounding 校验**：检查每句的引用是否真的在原文里有依据
3. 如果某句没有引用，或引用的文档没有依据，就标为 `unsupported`

这是 RAG 产品化里非常关键的一步——它把"看起来对"变成了"可以校验"。

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU cleared.')
    else: print('CUDA not available.')
except Exception as e: print(f'no torch ({e})')

In [ ]:
import json, math, re
from collections import Counter, defaultdict
from typing import List
import numpy as np
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)

## 1. 数据 + 检索（沿用）

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')
documents=[]; seen={}; eval_rows=[]
for item in raw_ds:
    c = item['context'].strip()
    if c not in seen:
        seen[c] = len(documents)
        documents.append({'doc_id': len(documents), 'text': c, 'title': item['title']})
    did = seen[c]
    if item['answers']['text']:
        eval_rows.append({'question': item['question'].strip(), 'gold_doc_id': did,
                          'reference_answer': item['answers']['text'][0].strip()})

def tokenize(t): return re.findall(r'[a-zA-Z0-9]+', t.lower())
class BM25:
    def __init__(self, c, k1=1.5, b=0.75):
        self.k1,self.b=k1,b; self.dl=np.array([len(t) for t in c],dtype=np.float32)
        self.avgdl=float(self.dl.mean()); self.N=len(c); self.post=defaultdict(dict)
        for did,toks in enumerate(c):
            for tok,f in Counter(toks).items(): self.post[tok][did]=f
        self.idf={t:math.log(1+(self.N-len(p)+0.5)/(len(p)+0.5)) for t,p in self.post.items()}
    def score(self,qt):
        s=np.zeros(self.N,dtype=np.float32)
        for tok in qt:
            if tok not in self.post: continue
            idf=self.idf[tok]
            for did,f in self.post[tok].items():
                dl=self.dl[did]; s[did]+=idf*(f*(self.k1+1))/(f+self.k1*(1-self.b+self.b*dl/self.avgdl))
        return s
bm25=BM25([tokenize(d['text']) for d in documents])
def get_emb(t,bs=16):
    out=[]
    for i in range(0,len(t),bs):
        r=client.embeddings.create(model=embedding_model, input=t[i:i+bs])
        out.extend([np.array(x.embedding,dtype=np.float32) for x in r.data])
    return np.vstack(out)
def l2n(m): n=np.linalg.norm(m,axis=1,keepdims=True); return m/np.clip(n,1e-12,None)
doc_embeddings=l2n(get_emb([d['text'] for d in documents]))
def hybrid(q,top_k=3,cand=15):
    lex=np.argsort(bm25.score(tokenize(q)))[::-1][:cand]
    qe=l2n(get_emb([q]))[0]; den=np.argsort(doc_embeddings@qe)[::-1][:cand]
    fused=defaultdict(float)
    for rk,i in enumerate(lex,1): fused[int(i)]+=1/(60+rk)
    for rk,i in enumerate(den,1): fused[int(i)]+=1/(60+rk)
    return [d for d,_ in sorted(fused.items(),key=lambda x:x[1],reverse=True)[:top_k]]
print('docs =', len(documents))

## 2. 让 LLM 输出带引用的结构化答案

我们要求 LLM 输出：
```
{
  "sentences": [
    {"text": "...", "citations": [doc_id, ...]},
    ...
  ]
}
```

然后我们再自己拼成用户可读的 `sentence [1][2]` 样式。

In [ ]:
def answer_with_citations(question, doc_ids):
    ctx='\n\n'.join([f'[doc_id={d}] title: {documents[d]["title"]}\n{documents[d]["text"]}' for d in doc_ids])
    sys = ('You answer the question using only the provided documents. '
           'Output strict JSON with one key "sentences". '
           'Each element has "text" (one sentence of your answer) and "citations" '
           '(a list of doc_id integers that directly support that sentence). '
           'If you cannot support a claim from the documents, do not include that sentence. '
           'Every sentence must have at least one citation.')
    usr = f'Question: {question}\n\nDocuments:\n{ctx}\n\nReturn only JSON.'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    raw = r.choices[0].message.content.strip()
    m = re.search(r'\{.*\}', raw, flags=re.S)
    if not m: raise ValueError(f'no json: {raw}')
    data = json.loads(m.group(0))
    return data, raw

## 3. Grounding 校验：每一句真的能在文档里找到依据吗？

两步校验：

- **轻校验（字面）**：句子里的关键 token 是否在引用的文档里出现
- **重校验（LLM judge）**：让 LLM 判断 "这句话是否被这个文档支持"

轻校验便宜、可解释；重校验更准，用作重要场景。

In [ ]:
STOPWORDS = set('a an the of in on at for to and or but is are was were be been being as by from with that this it its their they them he she his her him'.split())

def lexical_grounding(sentence: str, cited_doc_ids: List[int]) -> float:
    s_toks = [t for t in tokenize(sentence) if t not in STOPWORDS]
    if not s_toks: return 1.0
    union = set()
    for d in cited_doc_ids:
        union.update(tokenize(documents[d]['text']))
    overlap = sum(1 for t in s_toks if t in union)
    return overlap / len(s_toks)

def llm_grounding(sentence: str, cited_doc_ids: List[int]) -> bool:
    ctx = '\n\n'.join([f'[doc_id={d}]\n{documents[d]["text"]}' for d in cited_doc_ids])
    sys = 'You decide if the claim is directly supported by the provided documents. Respond with strict JSON {"supported": true|false}.'
    usr = f'Claim: {sentence}\n\nDocuments:\n{ctx}\n\nJSON:'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    raw = r.choices[0].message.content.strip()
    m = re.search(r'\{.*\}', raw, flags=re.S)
    if not m: return False
    try: return bool(json.loads(m.group(0)).get('supported', False))
    except Exception: return False

## 4. 组合：RAG with Citations + Grounding Check

In [ ]:
def rag_with_citations(question, top_k=3, lex_threshold=0.5):
    doc_ids = hybrid(question, top_k=top_k)
    data, _ = answer_with_citations(question, doc_ids)
    sentences = data.get('sentences', [])
    checked = []
    for s in sentences:
        text = s.get('text','').strip()
        cits = [int(c) for c in s.get('citations',[]) if isinstance(c, int) or str(c).isdigit()]
        cits = [c for c in cits if c in doc_ids]  # 必须是检索过的
        if not text or not cits:
            checked.append({'text': text, 'citations': cits, 'status': 'unsupported', 'lex': 0.0, 'llm_supported': False})
            continue
        lex = lexical_grounding(text, cits)
        if lex < lex_threshold:
            supported = llm_grounding(text, cits)
        else:
            supported = True
        checked.append({'text': text, 'citations': cits, 'lex': round(lex,3),
                        'llm_supported': supported,
                        'status': 'supported' if supported else 'unsupported'})
    return doc_ids, checked

def render_answer(checked):
    return '\n'.join([f'- {c["text"]}  [{",".join(str(x) for x in c["citations"])}]  ({c["status"]})' for c in checked])

## 5. 跑几条看效果

In [ ]:
for row in eval_rows[:3]:
    print('question :', row['question'])
    print('reference:', row['reference_answer'])
    doc_ids, checked = rag_with_citations(row['question'], top_k=3)
    print('retrieved doc_ids:', doc_ids)
    print('answer with citations:')
    print(render_answer(checked))
    n = len(checked); sup = sum(1 for c in checked if c['status']=='supported')
    print(f'grounding rate: {sup}/{n} = {0 if n==0 else sup/n:.2f}')
    print('-'*100)

## 6. 工程直觉

1. **让 LLM 输出 JSON 是最关键的一步**。字符串式"你自己加括号引用"非常容易出错。
2. 校验分两档：先做便宜的字面校验，只在不通过时再调 LLM judge。这能把成本压下来。
3. `citations 里必须是检索过的 doc_id` 这个过滤极其重要——LLM 偶尔会编造出一个不存在的 id。
4. 产品侧：`unsupported` 的句子要么不展示，要么用红色警示。**宁可少说，也不能带未支持的结论。**

下一课（也是这一系列的收尾）：**完整生产级 Pipeline + Ablation 消融实验**，把前面所有环节串起来，并用消融实验回答"到底哪一步贡献最大"。